# Multimodal Product Tagging with CLIP

E-commerce catalogs describe every product with several attributes — category, colour, gender, season — and filling them in is mostly manual. This notebook trains a single model to predict all four at once from a product's **image and title**, using CLIP's aligned vision-language embeddings.

It also looks at an honest failure mode: titles in real catalogs are often missing or noisy. Each attribute is scored with the title present and with it removed, so we can see which predictions hold up on the image alone.

**Setup:** turn on GPU and Internet (CLIP downloads on first use), and attach the *Fashion Product Images (Small)* dataset.

In [1]:
# CLIP lives in transformers; Kaggle ships torch/torchvision.
!pip -q install -U transformers
import torch, transformers
print('transformers', transformers.__version__, '| torch', torch.__version__,
      '| GPU available:', torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 106.7 MB/s eta 0:00:0000:010:01
transformers 5.12.0 | torch 2.10.0+cu128 | GPU available: True


## Configuration

Hyperparameters, the CLIP checkpoint, and the attributes to predict.

In [2]:
"""Central configuration for the multimodal product categorization project.

Every tunable lives here so the notebook, the CLI runner, and the Gradio app
all read the same knobs. The defaults are tuned for a *fast first run* on a
Kaggle GPU (frozen backbones, subsampled data); flip the marked flags for the
full training run that produces the full results.
"""
import os
import torch

# ---- Reproducibility ----
SEED = 42

# ---- Data: Fashion Product Images (Kaggle) ----
# One click on Kaggle: Add Data -> "Fashion Product Images (Small)"
# (paramaggarwal/fashion-product-images-small). The loader recursively finds
# styles.csv + the images/ folder, so the exact mount nesting doesn't matter.
DATA_ROOT = os.environ.get("DATA_ROOT", "/kaggle/input/fashion-product-images-small")
LABEL_COL = "articleType"    # what to predict. "articleType" (many, harder, best
                             # fusion story), "subCategory" (~45), "masterCategory" (~7, easy).
TOP_K_CATEGORIES = 20        # keep the K most frequent classes
MAX_PER_CLASS = 1500         # subsample each class -> fast first run
VAL_FRAC = 0.15
TEST_FRAC = 0.15

# If the dataset isn't found, generate a tiny synthetic set so the WHOLE pipeline still
# runs end-to-end (the numbers are meaningless). This lets
# you checking the code path before attaching the real dataset.
USE_SYNTHETIC_FALLBACK = True
SYNTHETIC_N = 1200

# ---- Text encoder ----
TEXT_MODEL = "distilbert-base-uncased"
MAX_TEXT_LEN = 48

# ---- Image encoder ----
IMAGE_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# ---- Training ----
# Defaults do a real FINE-TUNING run (unfrozen encoders) for strong results.
# For a quick check instead, set FREEZE_BACKBONE = True (trains only the head; weak numbers).
BATCH_SIZE = 32          # fits two fine-tuned backbones on a 16 GB T4; drop to 16 if you hit OOM
EPOCHS = 5
FREEZE_BACKBONE = False
HEAD_LR = 1e-3           # classifier head learns fast
BACKBONE_LR = 2e-5       # encoders fine-tune gently (unused when FREEZE_BACKBONE=True)
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

# Robustness experiment: product titles often spell out the category, so text alone
# nearly maxes out. To show the IMAGE earns its keep, we train the fusion model with
# random "missing title" augmentation, then test every arm with titles present vs.
# removed. Fusion should stay strong without titles (it falls back on the image),
# while text-only collapses.
TEXT_DROPOUT = 0.5       # fraction of titles blanked during fusion training

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACT_DIR = os.environ.get("ARTIFACT_DIR", "artifacts")

# ---- CLIP multi-task tagging ----
# One model reads image + title and predicts several catalog attributes
# at once (category, colour, gender, season) using CLIP's aligned embeddings.
CLIP_MODEL = "openai/clip-vit-base-patch32"
FREEZE_CLIP = True            # frozen CLIP features are strong AND fast; unfreeze for a final push
TAG_ATTRIBUTES = ["articleType", "baseColour", "gender", "season"]
TAG_TOPK = {"articleType": 20, "baseColour": 15, "gender": 5, "season": 5}
TAG_TEXT_MAXLEN = 32
TAG_HEAD_LR = 1e-3            # trunk + per-attribute heads learn fast
TAG_CLIP_LR = 1e-5           # CLIP fine-tunes gently (unused when FREEZE_CLIP=True)
TAG_EPOCHS = 6

print("Config ready  |  device:", DEVICE, "  |  CLIP:", CLIP_MODEL)

Config ready  |  device: cuda   |  CLIP: openai/clip-vit-base-patch32


## Data

Read the product metadata and images, keep the most common values per attribute, and wrap everything in a dataset that feeds CLIP's processor. If the dataset isn't attached, a small synthetic set lets the notebook still run end to end.

In [3]:
"""Multi-attribute Fashion data + CLIP-ready dataset.

We keep SEVERAL label columns (articleType,
baseColour, gender, season) and predict them all at once. Images + titles are
fed through a CLIPProcessor.
"""
import os
import glob
import random

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset

def _find(root, name):
    hits = glob.glob(os.path.join(root, "**", name), recursive=True)
    return hits[0] if hits else None

def _find_images_dir(root):
    for h in glob.glob(os.path.join(root, "**", "images"), recursive=True):
        if os.path.isdir(h):
            return h
    return None

def _read_styles(path):
    """Robust read of styles.csv (productDisplayName can contain commas)."""
    with open(path, encoding="utf-8") as f:
        header = [c.strip() for c in f.readline().rstrip("\n").split(",")]
        ncol = len(header)
        recs = [parts for line in f
                if len(parts := line.rstrip("\n").split(",", ncol - 1)) == ncol]
    return pd.DataFrame(recs, columns=header)

def _synthetic_multitag():
    """Fake multi-attribute data so the pipeline runs without the dataset (used when the dataset isn't attached)."""
    rng = random.Random(SEED)
    arts = ["Tshirts", "Shirts", "Watches", "Casual Shoes", "Handbags"]
    cols = ["Black", "Blue", "White", "Red", "Green"]
    gens = ["Men", "Women", "Unisex"]
    seas = ["Summer", "Winter", "Fall"]
    rows = []
    for i in range(SYNTHETIC_N):
        a, c, g, s = rng.choice(arts), rng.choice(cols), rng.choice(gens), rng.choice(seas)
        rows.append({"id": f"syn_{i}", "title": f"{g} {c} {a}",
                     "articleType": a, "baseColour": c, "gender": g, "season": s,
                     "image_path": None})
    df = pd.DataFrame(rows)
    maps = {a: {v: i for i, v in enumerate(sorted(df[a].unique()))} for a in TAG_ATTRIBUTES}
    return df, maps

def build_tagging_df():
    """Load Fashion data keeping all tagged attributes. Returns (df, label_maps, is_synthetic)."""
    styles = images_dir = None
    for root in [DATA_ROOT, "/kaggle/input"]:
        if root and os.path.exists(root):
            styles = styles or _find(root, "styles.csv")
            images_dir = images_dir or _find_images_dir(root)
            if styles and images_dir:
                break

    if not styles or not images_dir:
        print("=" * 72)
        print("  WARNING: Fashion dataset NOT FOUND -> using SYNTHETIC dummy data.")
        print("  Attach 'Fashion Product Images Small' (paramaggarwal) on Kaggle and re-run.")
        print("=" * 72)
        return (*_synthetic_multitag(), True)

    print(f"[data] Loaded real dataset from: {styles}")
    df = _read_styles(styles)
    need = ["id", "productDisplayName"] + TAG_ATTRIBUTES
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise RuntimeError(f"styles.csv missing columns {missing}; have {list(df.columns)}")
    df = df[need].rename(columns={"productDisplayName": "title"})
    for c in ["title"] + TAG_ATTRIBUTES:
        df = df[df[c].str.len() > 0]

    df["image_path"] = df["id"].map(lambda i: os.path.join(images_dir, f"{i}.jpg"))
    df = df[df.image_path.map(os.path.exists)]

    # Keep the top-K most frequent values per attribute (computed on the full data),
    # then drop rows that fall outside any attribute's top-K so every row is fully labelled.
    tops = {a: set(df[a].value_counts().head(TAG_TOPK[a]).index) for a in TAG_ATTRIBUTES}
    for a in TAG_ATTRIBUTES:
        df = df[df[a].isin(tops[a])]
    df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    maps = {a: {v: i for i, v in enumerate(sorted(df[a].unique()))} for a in TAG_ATTRIBUTES}
    return df, maps, False

def split_df(df):
    df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    n = len(df)
    n_test, n_val = int(n * TEST_FRAC), int(n * VAL_FRAC)
    return df.iloc[n_test + n_val:], df.iloc[n_test:n_test + n_val], df.iloc[:n_test]

class MultiTagDataset(Dataset):
    """Yields {pixel_values, input_ids, attention_mask, labels={attr: idx}} via the CLIP processor."""

    def __init__(self, df, processor, label_maps, train=False, text_dropout=0.0, blank_text=False):
        self.df = df.reset_index(drop=True)
        self.proc = processor
        self.maps = label_maps
        self.attrs = list(label_maps.keys())
        self.text_dropout = text_dropout
        self.blank_text = blank_text

    def __len__(self):
        return len(self.df)

    def _image(self, path):
        try:
            if path and os.path.exists(path):
                return Image.open(path).convert("RGB")
        except Exception:
            pass
        return Image.fromarray((np.random.rand(224, 224, 3) * 255).astype("uint8"))

    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = self._image(r.get("image_path", None))
        title = str(r["title"])
        if self.blank_text or (self.text_dropout > 0 and random.random() < self.text_dropout):
            title = ""
        pix = self.proc(images=img, return_tensors="pt")["pixel_values"][0]
        tok = self.proc(text=title, return_tensors="pt", padding="max_length",
                        max_length=TAG_TEXT_MAXLEN, truncation=True)
        labels = {a: torch.tensor(self.maps[a][r[a]], dtype=torch.long) for a in self.attrs}
        return {"pixel_values": pix, "input_ids": tok["input_ids"][0],
                "attention_mask": tok["attention_mask"][0], "labels": labels}

print("Data layer ready: build_tagging_df(), split_df(), MultiTagDataset")

Data layer ready: build_tagging_df(), split_df(), MultiTagDataset


## Model

CLIP gives aligned image and text embeddings. We concatenate them, pass them through a shared trunk, and branch into one linear head per attribute.

In [4]:
"""CLIP-backed multi-task tagging model.

CLIP gives us *aligned* text and image embeddings out of the box. We concatenate
them, pass through a shared trunk, and branch into one linear head per attribute
(category / colour / gender / season). Frozen CLIP + trained heads is fast and
strong; set FREEZE_CLIP=False to fine-tune.
"""
import torch
import torch.nn as nn
from transformers import CLIPModel

class CLIPMultiTask(nn.Module):
    def __init__(self, head_specs, freeze=True, trunk_dim=512, dropout=0.3):
        super().__init__()
        self.frozen = freeze
        self.clip = CLIPModel.from_pretrained(CLIP_MODEL)
        if freeze:
            for p in self.clip.parameters():
                p.requires_grad = False
        d = self.clip.config.projection_dim          # 512 for ViT-B/32
        self.trunk = nn.Sequential(
            nn.LayerNorm(2 * d), nn.Dropout(dropout),
            nn.Linear(2 * d, trunk_dim), nn.GELU(), nn.Dropout(dropout),
        )
        self.heads = nn.ModuleDict({a: nn.Linear(trunk_dim, n) for a, n in head_specs.items()})

    def _features(self, batch):
        # Use the full CLIP forward and read the projected embeds. This is stable
        # across transformers versions; get_image_features/get_text_features have
        # drifted (a recent build returns a BaseModelOutputWithPooling, not a tensor).
        out = self.clip(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            pixel_values=batch["pixel_values"],
            return_dict=True,
        )
        return out.image_embeds, out.text_embeds

    def forward(self, batch):
        if self.frozen:
            with torch.no_grad():
                img, txt = self._features(batch)
        else:
            img, txt = self._features(batch)
        h = self.trunk(torch.cat([img, txt], dim=1))
        return {a: head(h) for a, head in self.heads.items()}

print("Model ready: CLIPMultiTask")

Model ready: CLIPMultiTask


## Training and evaluation

In [5]:
"""Multi-task training / evaluation + the robustness test.

Loss is the sum of per-attribute cross-entropies. We report each attribute's
accuracy with the title present vs. removed — the multimodal model should hold up
on visual attributes (colour) even when the title is gone.
"""
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

def _to_device(batch):
    out = {}
    for k, v in batch.items():
        out[k] = {a: t.to(DEVICE) for a, t in v.items()} if k == "labels" else v.to(DEVICE)
    return out

def _loader(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS)

@torch.no_grad()
def evaluate_multitask(model, loader, attrs):
    model.eval()
    correct = {a: 0 for a in attrs}
    total = 0
    for batch in loader:
        batch = _to_device(batch)
        out = model(batch)
        total += next(iter(out.values())).size(0)
        for a in attrs:
            correct[a] += (out[a].argmax(1) == batch["labels"][a]).sum().item()
    return {a: correct[a] / max(total, 1) for a in attrs}

def train_multitask(model, train_ds, val_ds, attrs):
    train_loader, val_loader = _loader(train_ds, shuffle=True), _loader(val_ds)
    head_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("clip")]
    clip_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("clip")]
    groups = [{"params": head_params, "lr": TAG_HEAD_LR}]
    if clip_params:
        groups.append({"params": clip_params, "lr": TAG_CLIP_LR})
    opt = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    crit = nn.CrossEntropyLoss()

    best_mean, best_state = -1.0, None
    for epoch in range(1, TAG_EPOCHS + 1):
        model.train()
        running = 0.0
        for batch in train_loader:
            batch = _to_device(batch)
            opt.zero_grad()
            out = model(batch)
            loss = sum(crit(out[a], batch["labels"][a]) for a in attrs)
            loss.backward()
            opt.step()
            running += loss.item()
        accs = evaluate_multitask(model, val_loader, attrs)
        mean_acc = float(np.mean(list(accs.values())))
        print(f"epoch {epoch}/{TAG_EPOCHS}  loss {running / max(len(train_loader), 1):.3f}  "
              f"val_mean_acc {mean_acc:.3f}  " + "  ".join(f"{a[:4]}={accs[a]:.2f}" for a in attrs))
        if mean_acc > best_mean:
            best_mean = mean_acc
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
    if best_state:
        model.load_state_dict(best_state)
    return model

def run_tagging(train_df, val_df, test_df, processor, label_maps):
    attrs = list(label_maps.keys())
    head_specs = {a: len(label_maps[a]) for a in attrs}
    train_ds = MultiTagDataset(train_df, processor, label_maps, train=True, text_dropout=TEXT_DROPOUT)
    val_ds = MultiTagDataset(val_df, processor, label_maps)
    test_full = MultiTagDataset(test_df, processor, label_maps)
    test_blank = MultiTagDataset(test_df, processor, label_maps, blank_text=True)

    print(f"Training CLIP multi-task model (frozen={FREEZE_CLIP}) on {attrs} ...")
    model = CLIPMultiTask(head_specs, freeze=FREEZE_CLIP).to(DEVICE)
    model = train_multitask(model, train_ds, val_ds, attrs)

    acc_full = evaluate_multitask(model, _loader(test_full), attrs)
    acc_blank = evaluate_multitask(model, _loader(test_blank), attrs)
    res = pd.DataFrame([{
        "attribute": a, "n_classes": head_specs[a],
        "acc_with_title": round(acc_full[a], 4),
        "acc_no_title": round(acc_blank[a], 4),
    } for a in attrs])

    print("\nPer-attribute accuracy (one CLIP model, four heads):")
    print(res.to_string(index=False))
    print(f"\nMean accuracy with title:    {np.mean(list(acc_full.values())):.3f}")
    print(f"Mean accuracy, title removed: {np.mean(list(acc_blank.values())):.3f}  "
          "(visual attributes like colour survive on the image alone)")
    return res, model

print("Engine ready: run_tagging()")

Engine ready: run_tagging()


## Train and evaluate

Train the model, then score each attribute twice — with the title present, and with the title removed — to measure how much it leans on text vs. image.

In [6]:
import random
import numpy as np
import torch
from transformers import CLIPProcessor

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("Device:", DEVICE)

df, label_maps, synth = build_tagging_df()
print(f"{len(df)} products | synthetic={synth}")
for a, m in label_maps.items():
    print(f"  {a}: {len(m)} classes")
display(df.head())

train_df, val_df, test_df = split_df(df)
print(f"train/val/test = {len(train_df)}/{len(val_df)}/{len(test_df)}")

processor = CLIPProcessor.from_pretrained(CLIP_MODEL)
res, model = run_tagging(train_df, val_df, test_df, processor, label_maps)
res

Device: cuda
[data] Loaded real dataset from: /kaggle/input/datasets/paramaggarwal/fashion-product-images-small/styles.csv
30466 products | synthetic=False
  articleType: 20 classes
  baseColour: 15 classes
  gender: 5 classes
  season: 4 classes


,id,title,articleType,baseColour,gender,season,image_path
0,44672,Wills Lifestyle Women Blue Striped Formal Shirt,Shirts,Blue,Women,Winter,/kaggle/input/datasets/paramaggarwal/fashion-p...
1,45676,Tabac Original Edc Men Perfume,Perfume and Body Mist,Brown,Men,Spring,/kaggle/input/datasets/paramaggarwal/fashion-p...
2,10757,Disney Unisex Kids Princess Heart Pink Flip Flops,Flip Flops,Pink,Unisex,Winter,/kaggle/input/datasets/paramaggarwal/fashion-p...
3,33236,Arrow Men Grey Check Shirt,Shirts,Grey,Men,Summer,/kaggle/input/datasets/paramaggarwal/fashion-p...
4,44835,Puma Men Grey Training T-shirt,Tshirts,Grey,Men,Summer,/kaggle/input/datasets/paramaggarwal/fashion-p...


train/val/test = 21328/4569/4569


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

epoch 1/6  loss 2.331  val_mean_acc 0.901  arti=0.96  base=0.93  gend=0.99  seas=0.73
epoch 2/6  loss 1.794  val_mean_acc 0.909  arti=0.97  base=0.93  gend=0.99  seas=0.75
epoch 3/6  loss 1.671  val_mean_acc 0.908  arti=0.97  base=0.93  gend=0.99  seas=0.74
epoch 4/6  loss 1.586  val_mean_acc 0.920  arti=0.98  base=0.94  gend=0.99  seas=0.77
epoch 5/6  loss 1.521  val_mean_acc 0.921  arti=0.98  base=0.93  gend=0.99  seas=0.78
epoch 6/6  loss 1.477  val_mean_acc 0.921  arti=0.98  base=0.93  gend=0.99  seas=0.78

Per-attribute accuracy (one CLIP model, four heads):
  attribute  n_classes  acc_with_title  acc_no_title
articleType         20          0.9781        0.9271
 baseColour         15          0.9352        0.7437
     gender          5          0.9908        0.8932
     season          4          0.7851        0.7100

Mean accuracy with title:    0.922
Mean accuracy, title removed: 0.819  (visual attributes like colour survive on the image alone)


,attribute,n_classes,acc_with_title,acc_no_title
0,articleType,20,0.9781,0.9271
1,baseColour,15,0.9352,0.7437
2,gender,5,0.9908,0.8932
3,season,4,0.7851,0.7100


## Save the model

In [7]:
import os
import json

os.makedirs(ARTIFACT_DIR, exist_ok=True)
torch.save(model.state_dict(), os.path.join(ARTIFACT_DIR, "clip_multitask.pt"))
with open(os.path.join(ARTIFACT_DIR, "tag_label_maps.json"), "w") as f:
    json.dump({a: {str(k): v for k, v in m.items()} for a, m in label_maps.items()}, f)
res.to_csv(os.path.join(ARTIFACT_DIR, "tagging_results.csv"), index=False)
print("Saved ->", ARTIFACT_DIR, os.listdir(ARTIFACT_DIR))

Saved -> artifacts ['tagging_results.csv', 'clip_multitask.pt', 'tag_label_maps.json']


## Ideas to extend

- Fine-tune CLIP (`FREEZE_CLIP = False`) for a few more points.
- Add or swap attributes in `TAG_ATTRIBUTES` (e.g. `usage`, `subCategory`).
- Benchmark category accuracy against published numbers for this dataset.
- Wrap it in a small demo that shows every predicted tag for an uploaded product.